# CFNA one-click Colab/Jupyter setup, corpus build, training, and inference

Run cells from top to bottom. The first cell is intentionally self-contained for Google Colab: if the notebook is not already inside the repository, it clones the repo into `/content/nueronce`, installs dependencies, installs `cfna` editable, and changes the working directory to the repo root.

**Corpus policy:** the build uses only the sources registered in `cfna.corpus.sources.safe_commercial_sources()` and the corpus builder enforces allowed source hosts plus commercial-safe public-domain/CC0 license IDs. No blogs, social media, forums, scraped web, or copyrighted sources are used. The notebook also validates the manifest before training, so it will fail rather than train on an unapproved source/license.


In [ ]:
# 1) READY-TO-RUN SETUP FOR COLAB/JUPYTER
# If you forked the repo, change REPO_URL. Otherwise run this cell as-is.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/lmminier/nueronce.git"
BRANCH = None  # Example: "main". Leave as None to use the repo default branch.
REPO_DIR = Path("/content/nueronce") if Path("/content").exists() else Path.cwd()

# If this notebook is opened by itself in Colab/Drive, clone the code first.
if not (REPO_DIR / "pyproject.toml").exists():
    clone_cmd = ["git", "clone", "--depth", "1"]
    if BRANCH:
        clone_cmd += ["--branch", BRANCH]
    clone_cmd += [REPO_URL, str(REPO_DIR)]
    subprocess.run(clone_cmd, check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Colab normally already has numpy + torch, but requirements keeps local/Jupyter installs consistent.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-build-isolation"], check=True)

print("Ready. Working directory:", Path.cwd())


In [ ]:
# 2) Verify imports and hardware.
from pathlib import Path
import sys

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import torch
import cfna

print("repo:", REPO)
print("cfna:", cfna.__version__)
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
# 3) Inspect the only sources this notebook will use.
from cfna.corpus.sources import safe_commercial_sources, EXCLUDED_KINDS

sources = safe_commercial_sources()
for source in sources:
    print(f"{source.source_id}: {source.url} | {source.license_id} | bucket={source.bucket}")

print("\nExplicitly excluded source kinds:", ", ".join(EXCLUDED_KINDS))


In [ ]:
# 4) Build the allowed public-domain corpus.
# Downloads curated NLTK-hosted Project Gutenberg + US-government speech archives
# via raw.githubusercontent.com, then writes corpus/manifest.jsonl.
import subprocess

subprocess.run([sys.executable, "scripts/build_corpus.py", "--out", "corpus"], check=True)


In [ ]:
# 5) Verify every manifest record is allowed before training.
import json
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse

manifest_path = Path("corpus/manifest.jsonl")
records = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines() if line.strip()]
# raw.githubusercontent.com mirrors the NLTK corpora; www.gutenberg.org is the
# canonical public-domain book cache used to expand the literary corpus.
allowed_hosts = {"raw.githubusercontent.com", "www.gutenberg.org"}
allowed_license_ids = {"public-domain-us", "public-domain-usgov", "CC0-1.0"}

assert records, "corpus manifest is empty"
for record in records:
    assert record["bucket"] == "safe_commercial", record["document_id"]
    assert record["commercial_use"] is True, record["document_id"]
    assert record["attribution_required"] is False, record["document_id"]
    assert record["license_id"] in allowed_license_ids, record["document_id"]
    assert urlparse(record["source_locator"]).hostname in allowed_hosts, record["document_id"]

n_books = sum(r["document_type"] == "book" for r in records)
print(f"documents: {len(records)} ({n_books} books)")
print("splits:", dict(Counter(r["split"] for r in records)))
print("licenses:", dict(Counter(r["license_id"] for r in records)))
print("bytes:", sum(r["n_bytes"] for r in records))

In [ ]:
# 6) Train a checkpoint.
# Training time is the main driver of language quality: at ~1 minute the byte
# model has only seen a few dozen gradient steps and produces near-gibberish.
# For coherent words/phrases, train for at least 20-30 minutes (much faster on a
# Colab GPU than on CPU). Use --resume to keep extending the same checkpoint.
TRAIN_MINUTES = 20.0
CKPT = "checkpoints/cfna_chat.pt"

subprocess.run([
    sys.executable, "scripts/train_checkpoint.py",
    "--corpus", "corpus",
    "--minutes", str(TRAIN_MINUTES),
    "--out", CKPT,
    "--resume",
], check=True)

In [ ]:
# 7) Run the built-in chat/inference demo.
subprocess.run([
    sys.executable, "scripts/chat_demo.py",
    "--ckpt", CKPT,
    "--temp", "0.7",
    "--max-new", "120",
    "--seed", "0",
], check=True)


In [ ]:
# 8) Ask your own prompt from inside the notebook.
from cfna.chat import Conversation, load_checkpoint

model, ckpt = load_checkpoint(CKPT)
conversation = Conversation(model, system="A conversation.", temperature=0.7, max_new=120)

prompt = "Hello CFNA. Can you describe liberty in one paragraph?"
reply = conversation.say(prompt)
print("User:", prompt)
print("Assistant:", reply)
